# Ferrous + xarray — Mediterranean air temperature

This notebook fetches a tiny slice of CMIP6 surface air temperature using
[Ferrous](https://github.com/tham-le/ferrous), opens it with xarray, and
plots the first-month snapshot and the annual-mean trend.

The full file is **74 MB**; the slice is **~77 KB**. Everything below
runs against that 77 KB copy on disk.

In [ ]:
# Fetch the slice with Ferrous. Tolerant to repo-layout (running from the
# repo root) vs installed-on-PATH (production).
import os
import shutil
import subprocess


def find_ferrous() -> str:
    # 1) explicit override for CI / packaged deployments
    if env := os.environ.get("FERROUS_BIN"):
        return env
    # 2) system PATH
    if which := shutil.which("ferrous"):
        return which
    # 3) repo-local release build, resolved relative to this notebook so it
    # works from any cwd. Walk up from the notebook until we find Cargo.toml.
    here = os.path.abspath(os.path.dirname(__file__) if "__file__" in dir() else ".")
    for candidate in (here, os.path.dirname(here), os.path.dirname(os.path.dirname(here))):
        local = os.path.join(candidate, "target", "release", "ferrous")
        if os.path.isfile(local) and os.access(local, os.X_OK):
            return local
    raise RuntimeError(
        "ferrous binary not found. Build it with `cargo build --release` "
        "or set FERROUS_BIN=/path/to/ferrous."
    )


ferrous = find_ferrous()
out_path = "/tmp/ferrous_tas_med.nc"

if not os.path.exists(out_path):
    subprocess.run(
        [
            ferrous, "get",
            "--dataset-id",
            "CMIP6.ScenarioMIP.CNRM-CERFACS.CNRM-CM6-1.ssp245.r1i1p1f2.Amon.tas.gr.v20190219|esgf.ceda.ac.uk",
            "--variable", "tas",
            "--time-iso", "2020:2025",
            "--lat-deg", "30:46",
            "--lon-deg", "0:30",
            "--out", out_path,
        ],
        check=True,
    )

out_path

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt

ds = xr.open_dataset(out_path)
ds

## First-month snapshot

The time axis is monthly, starting in January 2020. Pull out the first
step and plot it as a filled contour.

In [ ]:
tas = ds['tas']
tas_c = tas - 273.15  # Kelvin -> Celsius

fig, ax = plt.subplots(figsize=(8, 4))
tas_c.isel(time=0).plot.contourf(ax=ax, levels=12, cmap='RdYlBu_r')
ax.set_title('Mediterranean surface air temperature, Jan 2020 (°C)')
plt.tight_layout()

## Annual-mean temperature trend

Ferrous copied the `calendar` and `units` attributes through from DAS, so
xarray has already decoded `time` to a real `datetime64` axis. That means
we can use `groupby('time.year')` directly — no manual coarsening, no
year-label guesswork.

In [ ]:
annual = tas_c.groupby('time.year').mean().mean(dim=['lat', 'lon'])

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(annual['year'].values, annual.values, marker='o')
ax.set(
    xlabel='year',
    ylabel='Mediterranean mean tas (°C)',
    title='Annual-mean surface air temperature, 2020-2025',
)
ax.grid(True, alpha=0.3)
plt.tight_layout()

## That's it

The whole notebook runs against 77 KB of local data. No ESGF login, no
4 GB download, no intake-esm dependency stack — just `ferrous get`
once, then vanilla xarray.

A second run of the notebook hits the local Ferrous cache and transfers
zero bytes from ESGF.